## Notebook 5 — Identity Diversity, Between-Cell Disagreement, and Local Conflict

NB4 tells us *whether* a population is impure.
This notebook asks *why*, using the identity GEPs NB1 classified and the labels NB2
assigned. It never touches the ROGUE calculation itself.

Four independent signals per label:

1. **ROGUE evidence** (NB4) — does the proposed GEP grouping improve purity?
2. **Within-cell identity diversity** — one identity program per cell, or several?
3. **Between-cell identity disagreement** — do cells favour different programs?
4. **Local interaction structure** — are co-active programs coordinated or antagonistic?

| Mechanism | Interpretation | CLANP action |
|---|---|---|
| `discrete_mixture` | cells individually specialised, but disagree | trust the split |
| `conflicted_continuum` | multi-program cells with local antagonism | do **not** force a split |
| `coordinated_hybrid` | multi-program cells, no conflict | retain composite identity |
| `uniform_identity_unexplained` | cells agree; identity does not explain low ROGUE | look elsewhere |
| `identity_unresolved` | too little usage on recognised identity GEPs | no biological verdict |

`rogue_verdict` and `mechanism_verdict` are kept in separate columns through to export;
`final_recommendation` is derived from both and never overwrites either.

**Inputs**
- `gep_program_class.tsv` (NB1)
- `per_gep_labels.tsv` (NB2)
- `rogue_gep_vs_baseline.csv` + `rogue_per_label_aggregate.csv` (NB4)
- cNMF usage matrices, the 5K HVG gene list (NB4), and the AnnData object

**Outputs**
- `per_cell_state.csv`
- `per_label_heterogeneity_verdict.csv`
- calibration + robustness diagnostics

In [ ]:
import os
import re
import glob
import json
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
rng = np.random.default_rng(0)
print("Imports OK.")

## 1. Configuration

Paths mirror NB1 / NB2 / NB4. Thresholds marked **CALIBRATE** come from the empirical
distributions in §3.5 and §6.4, not fixed defaults (§7.1).

In [ ]:
# upstream artefacts 
GSEA_DIR      = "combined-hnc_k3-35_gsea_celltype_results"
TREE_DIR      = "combined-hnc_k3-35_corr-tree_kl-divergence"
ROGUE_DIR     = "rogue_purity_results_per_gep"

PROGRAM_CLASS_PATH = os.path.join(GSEA_DIR,  "gep_program_class.tsv")         # NB1
PER_GEP_LABELS     = os.path.join(TREE_DIR,  "per_gep_labels.tsv")           # NB2
ROGUE_AUDIT_PATH   = os.path.join(ROGUE_DIR, "rogue_gep_vs_baseline.csv")     # NB4 long form
ROGUE_AGG_PATH     = os.path.join(ROGUE_DIR, "rogue_per_label_aggregate.csv") # NB4 aggregate
HVG_LIST_PATH      = os.path.join(ROGUE_DIR, "hvg_genes_n5000_seuratv3.txt")  # NB4 gene space

# cNMF 
CNMF_DIR          = "hnc_consensus_outputs_k3-35/combined_hnc_k_consensus_outputs"
CNMF_PREFIX       = "combined_hnc_k"
DENSITY_THRESHOLD = "0_01"
K_MIN, K_MAX      = 3, 35

# Expression / metadata 
ADATA_PATH         = "hnc_data/hnc_myeloid_2021.h5ad"
METADATA_PATH      = "hnc_data/hnc_myeloid_globalcluster4.csv"
METADATA_SEP       = ","
BARCODE_COL        = None
BASELINE_LABEL_COL = "global.cluster4"
SAMPLE_COL         = "orig.ident"

# Families (must match NB2 / NB4) 
CELLTYPE_FAMILIES = {
    "Monocyte":   ["Mono_CD14", "Mono_CD14_ID1", "Mono_CD14_IL1B", "Mono_CD14_THBS1",
                   "Mono_CD16", "Mono_Int", "Mono_TIL"],
    "DC":         ["cDC1_CLEC9A", "cDC2_CD1C", "cDC2_CD33", "mregDC_LAMP3"],
    "Macrophage": ["Mac_CXCL9", "Mac_IL1B", "Mac_IL1Bint", "Mac_SPP1"],
}
FAMILY_LABEL_SUFFIX = "_family"

# Program classes counted as identity GEPs (§3.2) 
IDENTITY_CLASSES  = {"identity_supported", "identity_ambiguous"}
EXCLUDED_CLASSES  = {"non_identity_unresolved"}
VALID_CLASSES     = IDENTITY_CLASSES | EXCLUDED_CLASSES

# Robust NB4 aggregation (§4.3)
DELTA_ROGUE_THRESHOLD = 0.05   # same band as NB4
MIN_SAMPLES_FOR_K     = 3 # min samples needed to score a (label,K,GEP) candidate

# Identity mass / shares (§5.2)
IDENTITY_MASS_MIN   = 0.30     # CALIBRATE from §3.5; below -> identity_unresolved
ACTIVE_SHARE_MIN    = 0.05     # min identity share to count a program as "active"

# Neighbourhoods (§6.2)
USE_EXISTING_EMBEDDING = False  # True -> use adata.obsm[EXISTING_EMBEDDING_KEY]
EXISTING_EMBEDDING_KEY = "X_pca_harmony"
N_PCS                   = 30
N_NEIGHBORS             = 40
N_NEIGHBORS_SWEEP       = [30, 40, 50]   # §6.2 sensitivity
NEIGHBORS_WITHIN_SAMPLE = True    # patient differences must not read as conflict
MIN_CELLS_PER_SAMPLE_FOR_KNN = 60

# CLR / local conflict (§6.3-§6.4)
CLR_EPSILON            = 1e-6
CLR_EPSILON_SWEEP      = [1e-5, 1e-6, 1e-7]   # zero-handling sensitivity
MIN_ACTIVE_PROGRAMS    = 2      # below this, a cell gets no interaction metrics
MIN_VALID_EDGES        = 1      # below -> quality flag, not "low conflict"
MIN_NEIGHBOR_VARIANCE  = 1e-12

# Conflict null (§5.5)
# CLR alone doesn't remove compositional closure; every conflict call is made on
# conflict_excess = observed - permuted-neighbour null.
USE_CLOSURE_OFFSET      = True
N_CONFLICT_PERMUTATIONS = 2
CONFLICT_NULL_SEED      = 7
CONFLICT_NEUTRAL        = 0.50   # closure-corrected conflict_fraction is centred here

# Per-cell state thresholds (§7.1)
# Percentiles within (K, sample) unless USE_FIXED_STATE_CUTOFFS.
USE_FIXED_STATE_CUTOFFS = False
DIVERSITY_HIGH_PCTL     = 60
DOMINANCE_HIGH_PCTL     = 60
CONFLICT_HIGH_PCTL      = 60
INTERACTION_MIN_PCTL    = 40     # below this, interaction too weak to call conflict either way
FIXED_DIVERSITY_HIGH    = 1.60   # used only if USE_FIXED_STATE_CUTOFFS
FIXED_DOMINANCE_HIGH    = 0.65
FIXED_CONFLICT_HIGH     = 0.35

# Per-label mechanism thresholds (§7.3)
LABEL_DIVERSITY_HIGH           = 1.50   # median effective identity programs
LABEL_JSD_HIGH                 = 0.15   # between-cell JSD (nats)
LABEL_CONFLICT_HIGH            = 0.10   # margin above CONFLICT_NEUTRAL when corrected
LABEL_IDENTITY_UNRESOLVED_MAX  = 0.50   # above -> identity_unresolved verdict

# Adjacent-K robustness (§10.3)
ADJACENT_K_OFFSETS = (-1, +1)

# Output
OUT_DIR = "clanp_heterogeneity_hnc"
os.makedirs(OUT_DIR, exist_ok=True)
PER_CELL_PATH  = os.path.join(OUT_DIR, "per_cell_state.csv")
PER_LABEL_PATH = os.path.join(OUT_DIR, "per_label_heterogeneity_verdict.csv")

print("Config loaded. Outputs ->", OUT_DIR)

#### 2. Module A — Validation and joins

Three jobs: load NB1's program classes, validate the `(K, GEP)` key space against
NB1/NB2/cNMF, and build a **robust** cross-sample ROGUE summary.

NB4's `rogue_per_label_aggregate.csv` picks `K_at_best` from the single highest-ROGUE
`(sample, K, GEP)` row, which can be one favourable patient. Here we re-derive the
selection from median evidence across samples, and build `rogue_verdict` explicitly
since NB4's aggregate has no recommendation column.

In [ ]:
def load_program_classes(path=PROGRAM_CLASS_PATH):
    """NB1 gep_program_class.tsv -> validated DataFrame with an is_identity flag."""
    df = pd.read_csv(path, sep='\t')
    required = {'k', 'gep', 'program_class', 'top_identity_label',
                'runner_up_ratio', 'n_significant_identity_sets', 'classification_reason'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns {sorted(missing)}. "
                         f"Got: {list(df.columns)}. Re-run NB1 §7.")
    df['k']   = df['k'].astype(int)
    df['gep'] = df['gep'].astype(str)
    bad = set(df['program_class']) - VALID_CLASSES
    if bad:
        raise ValueError(f"Unexpected program_class value(s): {bad}")
    df['is_identity'] = df['program_class'].isin(IDENTITY_CLASSES)
    return df


def usage_path_for_k(k):
    return os.path.join(
        CNMF_DIR, f"{CNMF_PREFIX}.usages.k_{k}.dt_{DENSITY_THRESHOLD}.consensus.txt")


def usage_gep_keys(k_values):
    """Authoritative (k, gep) universe, read straight from the cNMF usage files."""
    keys = set()
    for k in k_values:
        cols = pd.read_csv(usage_path_for_k(k), sep='\t', index_col=0, nrows=0).columns
        keys |= {(k, str(c)) for c in cols}
    return keys


def validate_gep_keys(program_class_df, labels_df, k_values):
    """Fail loudly if NB1 / NB2 / cNMF disagree about the (K, GEP) key space."""
    expected = usage_gep_keys(k_values)
    pc_keys  = set(zip(program_class_df['k'], program_class_df['gep']))
    lb_keys  = set(zip(labels_df['k'], labels_df['gep']))

    problems = []
    if program_class_df.duplicated(['k', 'gep']).any():
        problems.append("duplicate (k,gep) in gep_program_class.tsv")
    if labels_df.duplicated(['k', 'gep']).any():
        problems.append("duplicate (k,gep) in per_gep_labels.tsv")
    if expected - pc_keys:
        problems.append(f"{len(expected - pc_keys)} (k,gep) in usage files but absent from "
                        f"NB1: {sorted(expected - pc_keys)[:8]}")
    if pc_keys - expected:
        problems.append(f"{len(pc_keys - expected)} (k,gep) in NB1 but absent from usage files")
    if expected - lb_keys:
        problems.append(f"{len(expected - lb_keys)} (k,gep) in usage files but absent from "
                        f"NB2: {sorted(expected - lb_keys)[:8]}")
    if problems:
        raise ValueError("Module A key validation FAILED:\n  - " + "\n  - ".join(problems))
    print(f"\u2713 Key validation passed: {len(expected)} (K,GEP) keys agree across "
          f"cNMF usage files, NB1, and NB2.")
    return True

In [ ]:
program_class = load_program_classes()

labels_df = pd.read_csv(PER_GEP_LABELS, sep='\t')
labels_df['k']   = labels_df['k'].astype(int)
labels_df['gep'] = labels_df['gep'].astype(str)

k_values = sorted(set(program_class['k']) & set(labels_df['k']))
k_values = [k for k in k_values if K_MIN <= k <= K_MAX]

validate_gep_keys(program_class, labels_df, k_values)

print("\nProgram-class distribution:")
print(program_class['program_class'].value_counts().to_string())

n_ident_per_k = (program_class[program_class['is_identity']]
                 .groupby('k').size().rename('n_identity_programs'))
print("\nIdentity programs available per K (denominator for normalized entropy):")
print(n_ident_per_k.to_string())

if (n_ident_per_k < 2).any():
    bad_k = n_ident_per_k[n_ident_per_k < 2].index.tolist()
    print(f"\u26a0 K = {bad_k} have <2 identity GEPs -> entropy is degenerate there; "
          f"labels selecting those K will be flagged, not silently scored.")

#### 2.2 Robust cross-sample ROGUE summary (§4.3)

Aggregate the NB4 long-form audit by `membership_label + K + gep` **across samples**, then
pick the selected `(K, GEP)` from median evidence. The delta field depends on the baseline
type, exactly as in NB4 §8:

- `specific`  -> `delta_ROGUE` (vs. the subtype's own baseline)
- `subgroup` / `family` -> `delta_ROGUE_vs_max` (vs. the **best** constituent, the strict bar)

In [ ]:
def _delta_col_for(kind):
    return 'delta_ROGUE' if kind == 'specific' else 'delta_ROGUE_vs_max'


def build_robust_rogue_summary(audit_df,
                               min_samples=MIN_SAMPLES_FOR_K,
                               delta_threshold=DELTA_ROGUE_THRESHOLD):
    """Collapse NB4's long-form audit across samples into a robust per-label K/GEP pick."""
    df = audit_df.copy()
    df = df[df['baseline_group_type'].isin(['specific', 'subgroup', 'family'])]
    df['delta_used'] = [
        r['delta_ROGUE'] if r['baseline_group_type'] == 'specific' else r['delta_ROGUE_vs_max']
        for _, r in df[['baseline_group_type', 'delta_ROGUE', 'delta_ROGUE_vs_max']].iterrows()
    ]

    g = df.groupby(['membership_label', 'baseline_group_type', 'K', 'gep'], dropna=False)
    candidates = g.agg(
        median_delta=('delta_used', 'median'),
        delta_q25=('delta_used', lambda s: s.quantile(0.25)),
        delta_q75=('delta_used', lambda s: s.quantile(0.75)),
        n_samples=('delta_used', lambda s: s.notna().sum()),
        median_ROGUE=('ROGUE', 'median'),
        median_n_cells=('n_cells', 'median'),
        fraction_samples_positive=('delta_used',
                                   lambda s: float((s.dropna() > 0).mean())
                                   if s.notna().any() else np.nan),
        members_str=('members_str', 'first'),
    ).reset_index()
    candidates['delta_IQR'] = candidates['delta_q75'] - candidates['delta_q25']

    # Robust pick: highest median delta among candidates with enough sample support.
    ok = candidates[candidates['n_samples'] >= min_samples]
    fallback = False
    if ok.empty:
        print(f"\u26a0 No (label,K,GEP) candidate reaches {min_samples} samples. "
              f"Falling back to n_samples >= 1 and flagging every row.")
        ok, fallback = candidates.copy(), True

    ok = ok.sort_values(
        ['membership_label', 'median_delta', 'fraction_samples_positive', 'n_samples'],
        ascending=[True, False, False, False])
    selected = ok.groupby('membership_label', as_index=False).first()
    selected = selected.rename(columns={'K': 'K_selected', 'gep': 'GEP_selected'})

    # rogue_verdict: symmetric three-band ladder around +/- delta_threshold.
    def _verdict(d):
        if pd.isna(d):                    return 'no_baseline'
        if d >=  delta_threshold:         return 'purer'
        if d <= -delta_threshold:         return 'less_pure'
        return 'tie'
    selected['rogue_verdict'] = selected['median_delta'].map(_verdict)
    selected['rogue_selection_flag'] = 'low_sample_support' if fallback else ''
    selected.loc[selected['n_samples'] < min_samples, 'rogue_selection_flag'] = 'low_sample_support'
    return candidates, selected

In [ ]:
audit_df = pd.read_csv(ROGUE_AUDIT_PATH)
audit_df['gep'] = audit_df['gep'].astype(str)
audit_df['K']   = audit_df['K'].astype(int)
print(f"NB4 long-form audit: {len(audit_df)} rows, "
      f"{audit_df['membership_label'].nunique()} membership labels, "
      f"{audit_df['sample'].nunique()} samples")

nb4_agg = pd.read_csv(ROGUE_AGG_PATH)
if 'recommendation' in nb4_agg.columns:
    print("note: NB4 aggregate already carries a recommendation column; NB5 still builds "
          "rogue_verdict itself so the provenance is unambiguous.")

rogue_candidates, rogue_selected = build_robust_rogue_summary(audit_df)
rogue_candidates.to_csv(os.path.join(OUT_DIR, "rogue_candidates_by_label_K_GEP.csv"), index=False)

print(f"\nRobust selection: {len(rogue_selected)} labels")
print(rogue_selected['rogue_verdict'].value_counts().to_string())

# How often does robust selection disagree with NB4's single-best-row K_at_best?
_cmp = rogue_selected.merge(
    nb4_agg[['membership_label', 'K_at_best', 'GEP_at_best', 'sample_at_best']],
    on='membership_label', how='left')
_moved = _cmp[(_cmp['K_selected'] != _cmp['K_at_best'])]
print(f"\nRobust K differs from NB4 K_at_best for {len(_moved)}/{len(_cmp)} labels "
      f"-- these are the ones NB4 selected off a single favourable sample:")
if len(_moved):
    print(_moved[['membership_label', 'K_at_best', 'sample_at_best',
                  'K_selected', 'GEP_selected', 'n_samples',
                  'median_delta', 'fraction_samples_positive']]
          .head(15).to_string(index=False))

#### 3. Module B/C — Cell sets, identity mass, identity shares

**Three cell sets, one primary (§4.4).** The mechanism verdict is computed on **Cell Set A**
(the original label pool). Scoring only Cell Set B would bias toward dominance, since B is
*defined* by GEP assignment. B and A∩B are diagnostics only.

**Identity mass before renormalising (§5.2).** A cell that's 95% non-identity usage with the
remaining 5% split across two identity programs isn't a hybrid — it's unscoreable. Compute
mass first, gate on it, then close to shares.

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
meta = pd.read_csv(METADATA_PATH, sep=METADATA_SEP,
                   index_col=0 if BARCODE_COL is None else None)
if BARCODE_COL is not None:
    meta = meta.set_index(BARCODE_COL)
meta.index = meta.index.astype(str)

common = adata.obs_names.astype(str).intersection(meta.index)
adata = adata[common].copy()
adata.obs_names = adata.obs_names.astype(str)
meta = meta.loc[common]

cell_sample = adata.obs.loc[common, SAMPLE_COL].astype(str)
cell_orig_label = meta[BASELINE_LABEL_COL].astype(str)

hvg_genes = pd.read_csv(HVG_LIST_PATH, header=None)[0].astype(str).tolist()
hvg_genes = [g for g in hvg_genes if g in set(adata.var_names.astype(str))]
print(f"{len(common)} cells | {cell_sample.nunique()} samples | "
      f"{len(hvg_genes)} HVGs (cNMF substrate, from NB4)")
print(cell_orig_label.value_counts().to_string())

In [ ]:
_usage_cache = {}

def load_usage(k):
    """cNMF usage matrix at K, cells x GEPs, restricted to the analysed cells."""
    if k in _usage_cache:
        return _usage_cache[k]
    u = pd.read_csv(usage_path_for_k(k), sep='\t', index_col=0)
    u.index   = u.index.astype(str)
    u.columns = u.columns.astype(str)
    u = u.loc[u.index.intersection(common)]
    _usage_cache[k] = u
    return u


def identity_geps_at_k(k):
    """Ordered list of GEP ids at K whose program_class is an identity class (§3.2)."""
    sub = program_class[(program_class['k'] == k) & program_class['is_identity']]
    return sorted(sub['gep'].astype(str).tolist(), key=lambda g: (len(g), g))


def get_original_pool(membership_label, kind, members_str=None):
    """Cell Set A -- cells carrying the ORIGINAL annotation(s) behind this label (§4.4)."""
    if kind == 'specific':
        members = [membership_label]
    elif kind == 'subgroup':
        members = str(members_str).split('+') if members_str else []
    elif kind == 'family':
        fam = membership_label[:-len(FAMILY_LABEL_SUFFIX)]
        members = CELLTYPE_FAMILIES.get(fam, [])
    else:
        members = []
    members = [m for m in members if m]
    idx = cell_orig_label.index[cell_orig_label.isin(members)]
    return pd.Index(idx), members


def get_selected_gep_cells(k, gep):
    """Cell Set B -- cells whose argmax usage at K is this GEP."""
    u = load_usage(k)
    argmax = u.idxmax(axis=1).astype(str)
    return pd.Index(argmax.index[argmax == str(gep)])


def get_intersection_cells(pool_a, pool_b):
    return pool_a.intersection(pool_b)

In [ ]:
def restrict_identity_usages(k, cells):
    """Usage submatrix on identity GEPs only (§5.2 numerator)."""
    u = load_usage(k)
    cells = pd.Index(cells).intersection(u.index)
    ident = [g for g in identity_geps_at_k(k) if g in u.columns]
    return u.loc[cells, ident], u.loc[cells], ident


def calculate_identity_mass(u_ident, u_all):
    """identity_mass(c) = sum(identity usages) / sum(all usages). Computed BEFORE closure."""
    tot = u_all.sum(axis=1)
    tot = tot.replace(0, np.nan)
    return (u_ident.sum(axis=1) / tot).astype(float)


def close_identity_shares(u_ident):
    """p(c,j) = u(c,j) / sum_l u(c,l) over identity programs only (§5.3)."""
    s = u_ident.sum(axis=1).replace(0, np.nan)
    return u_ident.div(s, axis=0)

#### 3.5 Calibrate `IDENTITY_MASS_MIN` from the empirical distribution

Too low, and renormalisation manufactures hybrids out of noise; too high, and most cells
become unscoreable. Check the distribution below before trusting the default.

In [ ]:
_mass_rows = []
for k in sorted(rogue_selected['K_selected'].unique()):
    u_id, u_all, ident = restrict_identity_usages(k, common)
    if not ident:
        print(f"  K={k}: no identity GEPs -- skipped")
        continue
    m = calculate_identity_mass(u_id, u_all)
    _mass_rows.append(pd.DataFrame({'k': k, 'identity_mass': m.values,
                                    'n_identity_programs': len(ident)}))
mass_diag = pd.concat(_mass_rows, ignore_index=True) if _mass_rows else pd.DataFrame()

if len(mass_diag):
    print("identity_mass quantiles by selected K:")
    print(mass_diag.groupby('k')['identity_mass']
          .quantile([.1, .25, .5, .75, .9]).unstack().round(3).to_string())
    print(f"\nFraction of cells below IDENTITY_MASS_MIN={IDENTITY_MASS_MIN}: "
          f"{(mass_diag['identity_mass'] < IDENTITY_MASS_MIN).mean():.1%}")

    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.hist(mass_diag['identity_mass'].dropna(), bins=60, color='teal', alpha=.85)
    ax.axvline(IDENTITY_MASS_MIN, color='crimson', ls='--',
               label=f'IDENTITY_MASS_MIN={IDENTITY_MASS_MIN}')
    ax.set(xlabel='identity_mass', ylabel='# cells',
           title='Usage mass on identity GEPs (left of line -> identity_unresolved)')
    ax.legend(); plt.tight_layout(); plt.show()

#### 4. Module D — Within-cell diversity and between-cell disagreement

Three entropy representations are kept (§5.4). None works alone across K: raw effective
program count rises mechanically as more identity programs become available, so it can't
be the sole diversity metric.

In [ ]:
def calculate_entropy_metrics(p, n_identity_available):
    """p: cells x identity programs (shares). Returns entropy, effective count,
    normalized entropy, dominance per cell."""
    P = np.clip(p.values.astype(float), 0, None)
    with np.errstate(divide='ignore', invalid='ignore'):
        logP = np.where(P > 0, np.log(P), 0.0)
        H = -np.nansum(P * logP, axis=1)
    H = np.where(np.isfinite(H), H, np.nan)
    eff = np.exp(H)
    denom = np.log(n_identity_available) if n_identity_available > 1 else np.nan
    Hn = H / denom if denom and np.isfinite(denom) else np.full_like(H, np.nan)
    dom_share = np.nanmax(P, axis=1) if P.shape[1] else np.full(P.shape[0], np.nan)
    dom_gep = (p.columns.values[np.nanargmax(P, axis=1)]
               if P.shape[1] else np.array([None] * P.shape[0], dtype=object))
    n_active = (P >= ACTIVE_SHARE_MIN).sum(axis=1)
    return pd.DataFrame({
        'entropy': H,
        'effective_identity_programs': eff,
        'normalized_identity_entropy': Hn,
        'dominant_identity_share': dom_share,
        'dominant_identity_gep': dom_gep,
        'n_active_identity_programs': n_active,
    }, index=p.index)


def calculate_group_jsd(p):
    """Between-cell disagreement: JSD_group = H(mean shares) - mean_c H(shares).
    Zero when every cell has the same composition."""
    P = np.clip(p.values.astype(float), 0, None)
    P = P[np.isfinite(P).all(axis=1)]
    if P.shape[0] < 2 or P.shape[1] < 2:
        return np.nan
    def _H(v):
        v = np.clip(v, 0, None)
        s = v.sum()
        if s <= 0:
            return np.nan
        v = v / s
        nz = v[v > 0]
        return float(-(nz * np.log(nz)).sum())
    H_mean = _H(P.mean(axis=0))
    mean_H = float(np.nanmean([_H(row) for row in P]))
    return H_mean - mean_H


def summarize_dominant_geps(dominant_gep, min_frac=0.10):
    """Interpretable companions to JSD: entropy and concentration of dominant-GEP calls."""
    s = pd.Series(dominant_gep).dropna()
    if s.empty:
        return dict(dominant_gep_entropy=np.nan, largest_dominant_gep_fraction=np.nan,
                    n_dominant_geps_over_10pct=np.nan)
    frac = s.value_counts(normalize=True)
    nz = frac[frac > 0].values
    return dict(
        dominant_gep_entropy=float(-(nz * np.log(nz)).sum()),
        largest_dominant_gep_fraction=float(frac.iloc[0]),
        n_dominant_geps_over_10pct=int((frac >= min_frac).sum()),
    )

#### 5. Module E — Expression-space neighbourhoods, CLR, local conflict

**Expression space, not usage space (§6.2).** Usage-space neighbours would define the
neighbourhood using the very factor structure being evaluated. The graph uses the same 5K
HVG substrate cNMF saw, and (by default) is built **within sample** so patient differences
aren't read as conflict.

**CLR before correlation (§6.3).** Identity shares are compositional — raw correlations
among them are pushed negative by closure alone. Sequence: restrict -> mass -> close ->
CLR -> local estimation.

In [ ]:
_EMB_CACHE = {}

def _expression_embedding():
    """HVG PCA (or an existing batch-corrected embedding), computed once."""
    if 'emb' in _EMB_CACHE:
        return _EMB_CACHE['emb']
    if USE_EXISTING_EMBEDDING and EXISTING_EMBEDDING_KEY in adata.obsm:
        emb = np.asarray(adata.obsm[EXISTING_EMBEDDING_KEY])
        print(f"Using existing embedding '{EXISTING_EMBEDDING_KEY}' "
              f"({emb.shape[1]} dims) -- assumed batch-corrected.")
    else:
        sub = adata[:, hvg_genes].copy()
        sc.pp.normalize_total(sub, target_sum=1e4)
        sc.pp.log1p(sub)
        sc.pp.scale(sub, max_value=10)
        sc.tl.pca(sub, n_comps=min(N_PCS, min(sub.shape) - 1), svd_solver='arpack')
        emb = np.asarray(sub.obsm['X_pca'])
        del sub
        print(f"Built HVG PCA embedding: {emb.shape}")
    _EMB_CACHE['emb'] = emb
    return emb


def build_expression_neighbors(n_neighbors=N_NEIGHBORS):
    """Returns (neighbor_idx, valid_mask) in the global order of `common`. Samples with
    too few cells are marked invalid rather than borrowing neighbours across patients."""
    n = len(common)
    neighbor_idx = np.full((n, n_neighbors), -1, dtype=np.int64)
    valid = np.zeros(n, dtype=bool)
    pos = pd.Series(np.arange(n), index=common)

    if n_neighbors in _EMB_CACHE.get('knn', {}):
        return _EMB_CACHE['knn'][n_neighbors]
    emb_all = _expression_embedding()

    groups = ({s: idx for s, idx in cell_sample.groupby(cell_sample).groups.items()}
              if NEIGHBORS_WITHIN_SAMPLE else {'__all__': common})
    for s, idx in groups.items():
        rows = pos.loc[pd.Index(idx)].values
        if len(rows) < max(MIN_CELLS_PER_SAMPLE_FOR_KNN, n_neighbors):
            print(f"  \u26a0 sample {s}: {len(rows)} cells < required "
                  f"{max(MIN_CELLS_PER_SAMPLE_FOR_KNN, n_neighbors)} -- no neighbourhoods")
            continue
        nn = NearestNeighbors(n_neighbors=n_neighbors).fit(emb_all[rows])
        _, ind = nn.kneighbors(emb_all[rows])
        neighbor_idx[rows] = rows[ind]
        valid[rows] = True
    print(f"Neighbourhoods built for {valid.sum()}/{n} cells "
          f"(k={n_neighbors}, within_sample={NEIGHBORS_WITHIN_SAMPLE})")
    _EMB_CACHE.setdefault('knn', {})[n_neighbors] = (neighbor_idx, valid)
    return neighbor_idx, valid


def clr_transform(p, epsilon=CLR_EPSILON):
    """CLR(c,j) = log(p+eps) - mean_l log(p+eps). Compositional closure removed (§6.3)."""
    P = np.clip(p.values.astype(float), 0, None) + epsilon
    L = np.log(P)
    return pd.DataFrame(L - L.mean(axis=1, keepdims=True), index=p.index, columns=p.columns)

In [ ]:
def calculate_local_edges(p, clr, neighbor_idx, valid_mask, cell_positions,
                          corr_offset=None,
                          chunk=1500,
                          active_share_min=ACTIVE_SHARE_MIN,
                          min_active=MIN_ACTIVE_PROGRAMS,
                          min_var=MIN_NEIGHBOR_VARIANCE):
    """Per focal cell, pairwise identity-program correlations over its expression
    neighbours, weighted by focal-cell activity:

        weighted_edge(c,i,j) = [local_corr(c,i,j) - r0(i,j)] * sqrt(p(c,i) * p(c,j))
        conflict_fraction(c) = negative edge mass / total absolute edge mass

    `corr_offset` (r0) is the closure baseline from estimate_closure_offset(); without it
    conflict_fraction saturates near 1.0 for any P (§5.5). Cells failing the activity,
    neighbour, or variance requirements get NaN + a quality flag, never a silent 0.
    """
    n_cells, P = clr.shape
    out = {kk: np.full(n_cells, np.nan) for kk in
           ('conflict_fraction', 'interaction_strength', 'mean_abs_local_corr')}
    valid_edges = np.zeros(n_cells, dtype=int)
    flags = np.empty(n_cells, dtype=object); flags[:] = ''

    if P < 2:
        flags[:] = 'too_few_identity_programs'
        return pd.DataFrame({**out, 'valid_edge_count': valid_edges,
                             'interaction_flag': flags}, index=clr.index)

    Pshare = np.clip(p.values.astype(float), 0, None)
    Z_global = clr.values.astype(float)          # rows aligned to clr.index
    iu, ju = np.triu_indices(P, k=1)
    min_neighbors = max(5, int(0.25 * neighbor_idx.shape[1]))

    # global cell position -> local row index; neighbours outside the pool map to -1
    local_of_global = np.full(neighbor_idx.shape[0], -1, dtype=np.int64)
    local_of_global[cell_positions] = np.arange(n_cells)

    for start in range(0, n_cells, chunk):
        stop = min(start + chunk, n_cells)
        rows = np.arange(start, stop)
        gpos = cell_positions[rows]                    # global positions of focal cells
        ok = valid_mask[gpos]
        if not ok.any():
            flags[rows] = 'no_neighborhood'
            continue

        nbr_global = neighbor_idx[gpos]                # (m, kNN) global positions
        nbr_local = local_of_global[nbr_global]        # (m, kNN), -1 where unavailable
        present = nbr_local >= 0

        A = np.where(present[..., None], Z_global[np.clip(nbr_local, 0, None)], np.nan)
        n_ok = present.sum(axis=1)

        Am = np.nanmean(A, axis=1, keepdims=True)
        Ac = np.where(np.isfinite(A), A - Am, 0.0)
        counts = np.maximum(present.sum(axis=1) - 1, 1)[:, None, None]
        cov = np.einsum('mkp,mkq->mpq', Ac, Ac) / counts
        var = np.einsum('mpp->mp', cov)
        sd = np.sqrt(np.clip(var, 0, None))
        denom = sd[:, :, None] * sd[:, None, :]
        with np.errstate(divide='ignore', invalid='ignore'):
            corr = np.where(denom > 0, cov / denom, np.nan)
        corr = np.clip(corr, -1, 1)

        pw = Pshare[rows]                              # (m, P)
        act = pw >= active_share_min
        pair_active = act[:, iu] & act[:, ju]
        pair_var_ok = (var[:, iu] > min_var) & (var[:, ju] > min_var)
        c_ij = corr[:, iu, ju]
        if corr_offset is not None:
            c_ij = c_ij - np.asarray(corr_offset)[iu, ju][None, :]
        pair_ok = pair_active & pair_var_ok & np.isfinite(c_ij)

        w = np.sqrt(np.clip(pw[:, iu] * pw[:, ju], 0, None))
        edge = np.where(pair_ok, c_ij * w, np.nan)
        absmass = np.nansum(np.abs(edge), axis=1)
        negmass = np.nansum(np.where(edge < 0, -edge, 0.0), axis=1)
        n_valid = pair_ok.sum(axis=1)

        good = (ok & (n_ok >= min_neighbors) & (n_valid >= MIN_VALID_EDGES)
                & (absmass > 0) & (act.sum(axis=1) >= min_active))
        idx = rows[good]
        out['conflict_fraction'][idx] = negmass[good] / absmass[good]
        out['interaction_strength'][idx] = np.nanmean(np.abs(edge), axis=1)[good]
        out['mean_abs_local_corr'][idx] = np.nanmean(np.abs(np.where(pair_ok, c_ij, np.nan)),
                                                     axis=1)[good]
        valid_edges[rows] = n_valid

        for i, r in enumerate(rows):
            if good[i]:
                continue
            if not ok[i]:                        flags[r] = 'no_neighborhood'
            elif n_ok[i] < min_neighbors:        flags[r] = 'too_few_neighbors_in_pool'
            elif act[i].sum() < min_active:      flags[r] = 'too_few_active_programs'
            elif n_valid[i] < MIN_VALID_EDGES:   flags[r] = 'no_valid_edges'
            else:                                flags[r] = 'unstable_local_correlation'

    return pd.DataFrame({**out, 'valid_edge_count': valid_edges,
                         'interaction_flag': flags}, index=clr.index)


def summarize_conflict(edge_df):
    out = dict(
        median_conflict=float(np.nanmedian(edge_df['conflict_fraction'])),
        median_interaction_strength=float(np.nanmedian(edge_df['interaction_strength'])),
        interaction_missing_fraction=float(edge_df['conflict_fraction'].isna().mean()),
    )
    if 'conflict_fraction_raw' in edge_df.columns:
        out['median_conflict_uncorrected'] = float(np.nanmedian(edge_df['conflict_fraction_raw']))
        out['closure_offset_mean'] = float(np.nanmean(edge_df['closure_offset_mean']))
    return out

#### 5.5 Closure offset for conflict

CLR alone doesn't remove compositional closure: for **independent** parts, the residual
mean pairwise CLR correlation is exactly `-1/(P-1)`, and every pair comes out negative.

| P (identity programs) | mean pairwise CLR r |
|---|---|
| 3  | -0.50 |
| 5  | -0.25 |
| 8  | -0.14 |
| 20 | -0.05 |

`conflict_fraction` is sign-based, so at low P pure noise saturates near 1.0 — every
low-K label would look like a conflicted continuum. The fix has to happen on the
correlations, before signs are taken:

```
weighted_edge(c,i,j) = [ local_corr(c,i,j) - r0(i,j) ] * sqrt(p(c,i) p(c,j))
```

`r0` is estimated by permuting each program's share column within sample, re-closing,
re-CLR-ing, and recomputing local correlations on the same neighbourhood graph — this
destroys real co-variation but preserves P and the closure geometry. After correction,
`conflict_fraction` is centred at ~0.5 (`CONFLICT_NEUTRAL`); the per-label bar is
`CONFLICT_NEUTRAL + LABEL_CONFLICT_HIGH`.

§11.2 validates this on synthetic data:

| scenario | uncorrected (P=3) | corrected (P=3) |
|---|---|---|
| independent, no structure | 1.00 | 0.53 |
| real antagonism | 1.00 | 1.00 |
| real coordination | 0.00 | 0.00 |

`conflict_fraction_raw` is kept in the per-cell export so the correction stays auditable.

In [ ]:
def estimate_closure_offset(p, neighbor_idx, valid_mask, cell_positions,
                            n_perm=N_CONFLICT_PERMUTATIONS, seed=CONFLICT_NULL_SEED,
                            chunk=1500):
    """Per-program-pair closure baseline r0(i,j): mean local CLR correlation that
    survives when cross-program dependence is destroyed (each program's share column
    permuted within sample, re-closed, re-CLR'd, same neighbourhood graph). Subtracting
    r0 recentres conflict_fraction on ~0.5."""
    n_cells, P = p.shape
    if P < 2:
        return np.zeros((P, P))
    r = np.random.default_rng(seed)
    samples_local = cell_sample.loc[p.index].values
    acc = np.zeros((P, P)); cnt = np.zeros((P, P))

    local_of_global = np.full(neighbor_idx.shape[0], -1, dtype=np.int64)
    local_of_global[cell_positions] = np.arange(n_cells)

    for perm in range(max(1, n_perm)):
        Pm = np.clip(p.values.astype(float), 0, None).copy()
        for s_id in np.unique(samples_local):                 # permute within sample
            rows = np.where(samples_local == s_id)[0]
            if len(rows) < 5:
                continue
            for j in range(P):
                Pm[rows, j] = Pm[r.permutation(rows), j]
        tot = Pm.sum(axis=1, keepdims=True)
        Pm = np.divide(Pm, tot, out=np.full_like(Pm, np.nan), where=tot > 0)
        L = np.log(np.clip(Pm, 0, None) + CLR_EPSILON)
        Z = L - np.nanmean(L, axis=1, keepdims=True)

        for start in range(0, n_cells, chunk):
            stop = min(start + chunk, n_cells)
            rows = np.arange(start, stop)
            gpos = cell_positions[rows]
            ok = valid_mask[gpos]
            if not ok.any():
                continue
            nbr_local = local_of_global[neighbor_idx[gpos]]
            present = nbr_local >= 0
            A = np.where(present[..., None], Z[np.clip(nbr_local, 0, None)], np.nan)
            Am = np.nanmean(A, axis=1, keepdims=True)
            Ac = np.where(np.isfinite(A), A - Am, 0.0)
            counts = np.maximum(present.sum(axis=1) - 1, 1)[:, None, None]
            cov = np.einsum('mkp,mkq->mpq', Ac, Ac) / counts
            var = np.einsum('mpp->mp', cov)
            sd = np.sqrt(np.clip(var, 0, None))
            den = sd[:, :, None] * sd[:, None, :]
            with np.errstate(divide='ignore', invalid='ignore'):
                corr = np.where(den > 0, cov / den, np.nan)
            corr = np.clip(corr, -1, 1)[ok]
            acc += np.nansum(corr, axis=0)
            cnt += np.isfinite(corr).sum(axis=0)

    with np.errstate(invalid='ignore'):
        r0 = np.where(cnt > 0, acc / np.maximum(cnt, 1), 0.0)
    np.fill_diagonal(r0, 0.0)
    return r0

#### 6. Module F — Per-cell states, mechanism verdicts, reconciliation

**Thresholds are relative, not universal (§7.1).** Diversity, dominance, and conflict
cutoffs are percentiles computed *within (K, sample)* — higher K mechanically raises
diversity, and patients differ, so a fixed global cutoff would encode both as biology.

In [ ]:
def _within_group_percentile(values, k_series, sample_series):
    """Percentile rank of each value within its own (K, sample) stratum."""
    df = pd.DataFrame({'v': values, 'k': k_series, 's': sample_series})
    return (df.groupby(['k', 's'])['v']
              .rank(pct=True, na_option='keep') * 100).values


def assign_cell_state(cells_df):
    """Per-cell state. Identity mass gates everything; unmeasurable interaction never
    becomes 'low conflict'."""
    n = len(cells_df)
    state = np.array(['dominant_identity'] * n, dtype=object)
    reason = np.empty(n, dtype=object); reason[:] = ''

    if USE_FIXED_STATE_CUTOFFS:
        hi_div  = cells_df['effective_identity_programs'].values >= FIXED_DIVERSITY_HIGH
        hi_dom  = cells_df['dominant_identity_share'].values >= FIXED_DOMINANCE_HIGH
        hi_conf = cells_df['conflict_fraction'].values >= FIXED_CONFLICT_HIGH
        weak_int = np.zeros(n, dtype=bool)
    else:
        hi_div  = cells_df['diversity_percentile'].values >= DIVERSITY_HIGH_PCTL
        hi_dom  = cells_df['dominance_percentile'].values >= DOMINANCE_HIGH_PCTL
        hi_conf = cells_df['conflict_percentile'].values >= CONFLICT_HIGH_PCTL
        weak_int = cells_df['interaction_percentile'].values < INTERACTION_MIN_PCTL

    unresolved = (cells_df['identity_mass'].values < IDENTITY_MASS_MIN) | \
                 ~np.isfinite(cells_df['identity_mass'].values)
    no_interaction = ~np.isfinite(cells_df['conflict_fraction'].values)

    state[unresolved] = 'identity_unresolved'
    reason[unresolved] = f'identity_mass < {IDENTITY_MASS_MIN}'

    spec = (~unresolved) & (~hi_div) & hi_dom
    state[spec] = 'dominant_identity'
    reason[spec] = 'low diversity + high dominant share'

    multi = (~unresolved) & hi_div
    coord = multi & (~hi_conf) & (~weak_int) & (~no_interaction)
    confl = multi & hi_conf & (~weak_int) & (~no_interaction)
    unmeas = multi & (no_interaction | weak_int)

    state[coord]  = 'coordinated_multi_identity'
    reason[coord] = 'high diversity, adequate interaction strength, low conflict'
    state[confl]  = 'conflicted_multi_identity'
    reason[confl] = 'high diversity, adequate interaction strength, high conflict'
    state[unmeas] = 'multi_identity_interaction_unmeasured'
    reason[unmeas] = 'high diversity but local interaction unusable -- NOT scored as low conflict'

    leftover = (~unresolved) & (~hi_div) & (~hi_dom)
    state[leftover] = 'intermediate_identity'
    reason[leftover] = 'neither clearly specialised nor clearly multi-program'
    return state, reason


def assign_mechanism_verdict(agg):
    """Per-label mechanism verdict. Independent of ROGUE by construction -- ROGUE only
    enters in reconcile_with_rogue()."""
    if not np.isfinite(agg.get('identity_unresolved_fraction', np.nan)) or \
       agg['identity_unresolved_fraction'] > LABEL_IDENTITY_UNRESOLVED_MAX:
        return 'identity_unresolved', 'identity mass too low across the population'

    div  = agg.get('median_effective_programs', np.nan)
    jsd  = agg.get('between_cell_JSD', np.nan)
    # median_conflict is already closure-corrected (§5.5); its neutral point is 0.5.
    conf = agg.get('median_conflict', np.nan)
    conf_bar = (CONFLICT_NEUTRAL + LABEL_CONFLICT_HIGH if USE_CLOSURE_OFFSET
                else LABEL_CONFLICT_HIGH)

    if not np.isfinite(div) or not np.isfinite(jsd):
        return 'undetermined', 'diversity or disagreement not estimable'

    hi_div = div >= LABEL_DIVERSITY_HIGH
    hi_jsd = jsd >= LABEL_JSD_HIGH
    hi_cf  = np.isfinite(conf) and conf >= conf_bar

    if (not hi_div) and hi_jsd:
        return 'discrete_mixture', 'cells individually specialised but disagree'
    if hi_div and hi_jsd:
        return 'complex_multi_route', 'both within-cell diversity and between-cell disagreement high'
    if hi_div and (not hi_jsd):
        if not np.isfinite(conf):
            return 'multi_identity_conflict_unmeasured', 'diversity high; conflict not estimable'
        return ('conflicted_continuum', 'shared multi-program cells with local antagonism') \
            if hi_cf else ('coordinated_hybrid', 'shared multi-program cells without conflict')
    return 'uniform_identity_unexplained', 'cells agree on one program; identity does not explain impurity'


def reconcile_with_rogue(rogue_verdict, mechanism_verdict):
    """Combine the two INDEPENDENT verdicts. Both survive as separate columns."""
    m, r = mechanism_verdict, rogue_verdict
    if m in ('identity_unresolved', 'undetermined', 'multi_identity_conflict_unmeasured'):
        return 'no_identity_verdict -- investigate inputs'
    if r == 'purer':
        if m == 'discrete_mixture':      return 'split supported (mixture + purity gain)'
        if m == 'conflicted_continuum':  return 'annotate as transition -- discrete split discouraged'
        if m == 'coordinated_hybrid':    return 'retain composite identity despite purity gain'
        if m == 'complex_multi_route':   return 'split only with explicit subtype hypotheses'
        return 'purity gain not explained by identity usage -- check state/technical/sample'
    if r == 'less_pure':
        return f'do not refine ({m})'
    if r in ('tie', 'no_baseline'):
        if m == 'discrete_mixture':      return 'mixture present but purity gain unproven -- keep constituents separate'
        if m == 'conflicted_continuum':  return 'continuum -- keep as is, annotate transition'
        if m == 'coordinated_hybrid':    return 'hybrid state -- retain composite label'
        return 'no action -- no purity or mechanism evidence'
    return 'no action'

#### 7. Run the pipeline

Labels are processed grouped by `K_selected` so each usage matrix and neighbour graph is
built once. Everything below is Cell Set A; B and A∩B are recorded as diagnostics only.

In [ ]:
neighbor_idx, neighbor_valid = build_expression_neighbors(N_NEIGHBORS)
global_pos = pd.Series(np.arange(len(common)), index=common)

In [ ]:
def run_label(row, n_neighbors_label=None, clr_epsilon=CLR_EPSILON):
    """Compute per-cell metrics + the per-label aggregate for one membership_label."""
    label   = row['membership_label']
    kind    = row['baseline_group_type']
    k       = int(row['K_selected'])
    gep_sel = str(row['GEP_selected'])

    pool_a, members = get_original_pool(label, kind, row.get('members_str'))
    u = load_usage(k)
    pool_a = pool_a.intersection(u.index)
    pool_b = get_selected_gep_cells(k, gep_sel)
    inter  = get_intersection_cells(pool_a, pool_b)

    ident = identity_geps_at_k(k)
    n_ident_avail = len(ident)
    if len(pool_a) < 10 or n_ident_avail < 2:
        return None, dict(label=label, label_kind=kind, K_selected=k, GEP_selected=gep_sel,
                          n_cells_A=len(pool_a), n_identity_programs_available=n_ident_avail,
                          quality_flags='insufficient_cells_or_identity_programs')

    u_id, u_all, ident = restrict_identity_usages(k, pool_a)
    mass = calculate_identity_mass(u_id, u_all)
    p    = close_identity_shares(u_id)
    keep = p.notna().all(axis=1)
    p, mass = p[keep], mass[keep]
    if len(p) < 10:
        return None, dict(label=label, label_kind=kind, K_selected=k, GEP_selected=gep_sel,
                          n_cells_A=len(p), quality_flags='no_cells_with_identity_usage')

    div = calculate_entropy_metrics(p, n_ident_avail)
    clr = clr_transform(p, epsilon=clr_epsilon)

    if n_neighbors_label is None:
        nidx, nval = neighbor_idx, neighbor_valid
    else:
        nidx, nval = build_expression_neighbors(n_neighbors_label)
    cpos = global_pos.loc[p.index].values
    r0 = (estimate_closure_offset(p, nidx, nval, cpos)
          if USE_CLOSURE_OFFSET else None)
    edges = calculate_local_edges(p, clr, nidx, nval, cpos, corr_offset=r0)
    raw  = calculate_local_edges(p, clr, nidx, nval, cpos, corr_offset=None)
    edges['conflict_fraction_raw'] = raw['conflict_fraction']
    edges['closure_offset_mean'] = (float(np.nanmean(r0[np.triu_indices(len(ident), 1)]))
                                    if r0 is not None and len(ident) > 1 else np.nan)

    cells = pd.concat([div, edges], axis=1)
    cells['identity_mass'] = mass
    cells['cell_id']       = cells.index
    cells['sample']        = cell_sample.loc[cells.index].values
    cells['original_label']= cell_orig_label.loc[cells.index].values
    cells['membership_label'] = label
    cells['label_kind']    = kind
    cells['K']             = k
    cells['n_identity_programs_available'] = n_ident_avail
    cells['in_selected_gep'] = cells.index.isin(pool_b)

    # Percentiles within (K, sample) -- §7.1
    cells['diversity_percentile']   = _within_group_percentile(
        cells['effective_identity_programs'], cells['K'], cells['sample'])
    cells['dominance_percentile']   = _within_group_percentile(
        cells['dominant_identity_share'], cells['K'], cells['sample'])
    cells['conflict_percentile']    = _within_group_percentile(
        cells['conflict_fraction'], cells['K'], cells['sample'])
    cells['interaction_percentile'] = _within_group_percentile(
        cells['interaction_strength'], cells['K'], cells['sample'])

    state, reason = assign_cell_state(cells)
    cells['cell_state']  = state
    cells['state_reason']= reason
    cells['quality_flags'] = np.where(cells['interaction_flag'].astype(str) != '',
                                      cells['interaction_flag'], '')

    scored = cells[cells['cell_state'] != 'identity_unresolved']
    p_scored = p.loc[scored.index]
    agg = dict(
        label=label, label_kind=kind, K_selected=k, GEP_selected=gep_sel,
        n_cells_A=len(cells), n_cells_B=len(pool_b), n_cells_AB=len(inter),
        n_samples=int(cells['sample'].nunique()),
        n_identity_programs_available=n_ident_avail,
        members_str=row.get('members_str', ''),
        identity_unresolved_fraction=float((cells['cell_state'] == 'identity_unresolved').mean()),
        median_effective_programs=float(np.nanmedian(scored['effective_identity_programs']))
            if len(scored) else np.nan,
        median_normalized_entropy=float(np.nanmedian(scored['normalized_identity_entropy']))
            if len(scored) else np.nan,
        between_cell_JSD=calculate_group_jsd(p_scored),
        **summarize_dominant_geps(scored['dominant_identity_gep'] if len(scored) else []),
        **summarize_conflict(scored if len(scored) else cells),
        fraction_conflicted=float((cells['cell_state'] == 'conflicted_multi_identity').mean()),
        fraction_coordinated=float((cells['cell_state'] == 'coordinated_multi_identity').mean()),
        fraction_dominant=float((cells['cell_state'] == 'dominant_identity').mean()),
        fraction_interaction_unmeasured=float(
            (cells['cell_state'] == 'multi_identity_interaction_unmeasured').mean()),
    )
    # Per-sample reproducibility of the diversity signal
    per_sample_div = scored.groupby('sample')['effective_identity_programs'].median()
    agg['sample_consistency'] = (float((per_sample_div >= LABEL_DIVERSITY_HIGH).mean())
                                 if len(per_sample_div) else np.nan)
    return cells, agg

In [ ]:
per_cell_frames, per_label_rows = [], []
for _, row in rogue_selected.iterrows():
    cells, agg = run_label(row)
    if cells is not None:
        per_cell_frames.append(cells)
    agg = dict(agg)
    agg.update({
        'rogue_verdict': row['rogue_verdict'],
        'median_delta': row['median_delta'],
        'delta_IQR': row['delta_IQR'],
        'fraction_samples_positive': row['fraction_samples_positive'],
        'n_samples_rogue': row['n_samples'],
        'rogue_selection_flag': row['rogue_selection_flag'],
    })
    per_label_rows.append(agg)
    print(f"  {row['membership_label']:<40} K={row['K_selected']:<3} "
          f"GEP={row['GEP_selected']:<4} cells_A={agg.get('n_cells_A', 0)}")

per_cell_state = (pd.concat(per_cell_frames, ignore_index=False)
                  if per_cell_frames else pd.DataFrame())
per_label = pd.DataFrame(per_label_rows)
print(f"\nScored {len(per_label)} labels, {len(per_cell_state)} (cell, label) rows.")

## 8. Silent-failure check 2 and verdict assembly

A cell with too few neighbours, too few active programs, or unstable correlations
produces *no* conflict estimate. If that silently became `conflict = 0`, every
unmeasurable cell would read as a coordinated hybrid.

In [ ]:
if len(per_cell_state):
    flag_counts = (per_cell_state['interaction_flag'].replace('', 'ok')
                   .value_counts())
    print("Interaction quality (Silent-failure check 2):")
    print(flag_counts.to_string())
    unmeasured = float((per_cell_state['conflict_fraction'].isna()).mean())
    print(f"\n{unmeasured:.1%} of cells have NO conflict estimate. These are excluded from "
          f"conflict medians and routed to 'multi_identity_interaction_unmeasured' or a "
          f"quality flag -- never counted as low conflict.")
    if unmeasured > 0.5:
        print("\u26a0 Over half of cells are unmeasurable. Fix the neighbourhood / activity "
              "settings before interpreting any conflict-based verdict.")
    print("\nCell-state composition overall:")
    print(per_cell_state['cell_state'].value_counts(normalize=True).round(3).to_string())

    # Closure diagnostic (§5.5): how much conflict was geometry rather than biology?
    if USE_CLOSURE_OFFSET and 'conflict_fraction_raw' in per_cell_state.columns:
        by_p = (per_cell_state.groupby('n_identity_programs_available')
                [['conflict_fraction_raw', 'conflict_fraction', 'closure_offset_mean']].median())
        by_p['analytic_closure_r'] = -1.0 / (by_p.index.values - 1)
        print("\nUncorrected vs closure-corrected conflict, by number of identity programs:")
        print(by_p.round(3).to_string())
        print("If conflict_fraction_raw tracks -1/(P-1) and falls toward 0.5 after correction, "
              "the uncorrected statistic was measuring simplex geometry, not antagonism.")

In [ ]:
mech = per_label.apply(lambda r: assign_mechanism_verdict(r.to_dict()), axis=1, result_type='expand')
per_label['mechanism_verdict'] = mech[0]
per_label['mechanism_reason']  = mech[1]
per_label['final_recommendation'] = [
    reconcile_with_rogue(r['rogue_verdict'], r['mechanism_verdict'])
    for _, r in per_label.iterrows()
]

def _confidence(r):
    score = 0
    if r.get('n_samples_rogue', 0) >= MIN_SAMPLES_FOR_K:                     score += 1
    if pd.notna(r.get('fraction_samples_positive')) and \
       (r['fraction_samples_positive'] >= 0.7 or r['fraction_samples_positive'] <= 0.3): score += 1
    if pd.notna(r.get('identity_unresolved_fraction')) and \
       r['identity_unresolved_fraction'] < 0.25:                             score += 1
    if pd.notna(r.get('fraction_interaction_unmeasured')) and \
       r['fraction_interaction_unmeasured'] < 0.25:                          score += 1
    if pd.notna(r.get('sample_consistency')) and \
       (r['sample_consistency'] >= 0.7 or r['sample_consistency'] <= 0.3):   score += 1
    return ['very_low', 'low', 'low', 'moderate', 'high', 'high'][score]

per_label['verdict_confidence'] = per_label.apply(_confidence, axis=1)

print("Mechanism verdicts:")
print(per_label['mechanism_verdict'].value_counts().to_string())
print("\nROGUE vs mechanism (the discordant cells are the point of this notebook):")
print(pd.crosstab(per_label['rogue_verdict'], per_label['mechanism_verdict']).to_string())

## 9. Adjacent-K consistency (§10.3)

Every headline conclusion must survive a nearby resolution. Higher K mechanically raises
diversity, so a verdict that flips between K and K±1 is a factorisation artefact.

In [ ]:
adjacent_rows = []
for _, row in rogue_selected.iterrows():
    base_k = int(row['K_selected'])
    base_mech = per_label.loc[per_label['label'] == row['membership_label'],
                              'mechanism_verdict']
    base_mech = base_mech.iloc[0] if len(base_mech) else None
    agree, tested = 0, 0
    for off in ADJACENT_K_OFFSETS:
        k2 = base_k + off
        if k2 not in k_values:
            continue
        cand = rogue_candidates[(rogue_candidates['membership_label'] == row['membership_label'])
                                & (rogue_candidates['K'] == k2)]
        if cand.empty:
            continue
        best = cand.sort_values('median_delta', ascending=False).iloc[0]
        alt = row.copy()
        alt['K_selected'], alt['GEP_selected'] = k2, str(best['gep'])
        _, agg2 = run_label(alt)
        if agg2 is None or 'median_effective_programs' not in agg2:
            continue
        tested += 1
        if assign_mechanism_verdict(agg2)[0] == base_mech:
            agree += 1
    adjacent_rows.append({'label': row['membership_label'],
                          'adjacent_K_tested': tested,
                          'adjacent_K_agree': agree,
                          'adjacent_K_consistency': (agree / tested) if tested else np.nan})

adjacent_df = pd.DataFrame(adjacent_rows)
per_label = per_label.merge(adjacent_df, on='label', how='left')
print(per_label[['label', 'K_selected', 'mechanism_verdict',
                 'adjacent_K_tested', 'adjacent_K_consistency']].to_string(index=False))

#### 10. Exports

`rogue_verdict`, `mechanism_verdict`, and `final_recommendation` stay as three separate
columns so a reviewer can see when purity and mechanism disagree.

In [ ]:
PER_CELL_COLS = [
    'cell_id', 'sample', 'original_label', 'membership_label', 'label_kind', 'K',
    'identity_mass', 'n_identity_programs_available', 'n_active_identity_programs',
    'entropy', 'effective_identity_programs', 'normalized_identity_entropy',
    'diversity_percentile',
    'dominant_identity_gep', 'dominant_identity_share',
    'conflict_fraction', 'conflict_fraction_raw', 'closure_offset_mean',
    'interaction_strength', 'valid_edge_count',
    'in_selected_gep', 'cell_state', 'state_reason', 'quality_flags',
]
PER_LABEL_COLS = [
    'label', 'label_kind', 'K_selected', 'GEP_selected', 'n_samples', 'members_str',
    'rogue_verdict', 'median_delta', 'fraction_samples_positive', 'delta_IQR',
    'median_effective_programs', 'median_normalized_entropy', 'identity_unresolved_fraction',
    'between_cell_JSD', 'dominant_gep_entropy', 'largest_dominant_gep_fraction',
    'n_dominant_geps_over_10pct',
    'median_conflict', 'median_conflict_uncorrected', 'closure_offset_mean',
    'median_interaction_strength', 'fraction_conflicted',
    'fraction_coordinated', 'fraction_interaction_unmeasured',
    'mechanism_verdict', 'mechanism_reason', 'final_recommendation',
    'verdict_confidence', 'adjacent_K_consistency', 'sample_consistency',
    'n_cells_A', 'n_cells_B', 'n_cells_AB', 'rogue_selection_flag',
]

if len(per_cell_state):
    out_cells = per_cell_state.reindex(columns=PER_CELL_COLS)
    out_cells.to_csv(PER_CELL_PATH, index=False)
    print(f"Wrote {PER_CELL_PATH}  ({len(out_cells)} rows)")

out_label = per_label.reindex(columns=PER_LABEL_COLS)
out_label.to_csv(PER_LABEL_PATH, index=False)
print(f"Wrote {PER_LABEL_PATH}  ({len(out_label)} rows)")
out_label

## 11. Robustness: neighbourhood size and zero handling

Acceptance criterion: key verdicts stable across at least two neighbourhood sizes. This
cell is expensive — restrict `ROBUSTNESS_LABELS` to the headline labels.

In [ ]:
ROBUSTNESS_LABELS = per_label.sort_values('median_delta', ascending=False)['label'].head(5).tolist()
print("Robustness labels:", ROBUSTNESS_LABELS)

rob_rows = []
for lbl in ROBUSTNESS_LABELS:
    row = rogue_selected[rogue_selected['membership_label'] == lbl]
    if row.empty:
        continue
    row = row.iloc[0]
    for nn in N_NEIGHBORS_SWEEP:
        for eps in CLR_EPSILON_SWEEP:
            _, agg2 = run_label(row, n_neighbors_label=nn, clr_epsilon=eps)
            if agg2 is None or 'median_conflict' not in agg2:
                continue
            rob_rows.append({'label': lbl, 'n_neighbors': nn, 'clr_epsilon': eps,
                             'median_conflict': agg2['median_conflict'],
                             'median_effective_programs': agg2['median_effective_programs'],
                             'between_cell_JSD': agg2['between_cell_JSD'],
                             'mechanism_verdict': assign_mechanism_verdict(agg2)[0]})

robustness_df = pd.DataFrame(rob_rows)
if len(robustness_df):
    robustness_df.to_csv(os.path.join(OUT_DIR, "robustness_neighbors_epsilon.csv"), index=False)
    stab = (robustness_df.groupby('label')['mechanism_verdict']
            .agg(lambda s: s.value_counts(normalize=True).iloc[0]))
    print("\nMechanism-verdict stability across neighbourhood size x CLR epsilon:")
    print(stab.round(2).to_string())
    unstable = stab[stab < 1.0]
    if len(unstable):
        print(f"\n\u26a0 Unstable verdicts: {list(unstable.index)} -- mark these sample-specific "
              f"or parameter-sensitive in NB5 rather than reporting them as findings.")
    print(robustness_df.to_string(index=False))

#### 12. Redundancy analysis

High overall correlation between ROGUE and diversity doesn't make the mechanism axis
useless — the question is whether it *changes the interpretation of low-ROGUE labels
reproducibly*. Look at the discordant labels, not the scatter.

In [ ]:
_r = per_label.dropna(subset=['median_delta', 'median_effective_programs'])
if len(_r) >= 4:
    rho, pv = spearmanr(_r['median_delta'], _r['median_effective_programs'])
    print(f"Spearman(median_delta, median_effective_programs) = {rho:.3f} (p={pv:.3g}), "
          f"n={len(_r)}")
    print("Stratified by label_kind:")
    for kind, sub in _r.groupby('label_kind'):
        if len(sub) >= 4:
            rr, pp = spearmanr(sub['median_delta'], sub['median_effective_programs'])
            print(f"  {kind:<10} rho={rr:6.3f}  p={pp:.3g}  n={len(sub)}")

discordant = per_label[
    ((per_label['rogue_verdict'] == 'purer') &
     per_label['mechanism_verdict'].isin(['conflicted_continuum', 'coordinated_hybrid'])) |
    ((per_label['rogue_verdict'].isin(['tie', 'less_pure'])) &
     (per_label['mechanism_verdict'] == 'discrete_mixture'))
]
print(f"\n{len(discordant)} discordant label(s) -- ROGUE and mechanism imply different actions:")
if len(discordant):
    print(discordant[['label', 'rogue_verdict', 'mechanism_verdict',
                      'final_recommendation', 'verdict_confidence']].to_string(index=False))
    discordant.to_csv(os.path.join(OUT_DIR, "discordant_labels.csv"), index=False)
else:
    print("(none -- report this honestly in NB5; it means the second axis added no reversal "
          "on this dataset at these thresholds)")

#### 13. Quadrant figure

In [ ]:
if len(per_label):
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    colors = {'discrete_mixture': '#2b6cb0', 'conflicted_continuum': '#c53030',
              'coordinated_hybrid': '#2f855a', 'uniform_identity_unexplained': '#718096',
              'complex_multi_route': '#805ad5', 'identity_unresolved': '#cbd5e0'}
    for mech, sub in per_label.groupby('mechanism_verdict'):
        ax.scatter(sub['median_effective_programs'], sub['between_cell_JSD'],
                   s=60 + 400 * sub['fraction_conflicted'].fillna(0),
                   c=colors.get(mech, '#a0aec0'), alpha=.8, edgecolor='k',
                   linewidth=.6, label=mech)
    ax.axvline(LABEL_DIVERSITY_HIGH, ls='--', c='grey', lw=1)
    ax.axhline(LABEL_JSD_HIGH, ls='--', c='grey', lw=1)
    ax.set(xlabel='within-cell diversity (median effective identity programs)',
           ylabel='between-cell disagreement (group JSD)',
           title='Heterogeneity decomposition (§6)\npoint size = fraction of conflicted cells')
    ax.legend(fontsize=8, frameon=False, loc='best')
    for _, r in per_label.iterrows():
        if pd.notna(r['median_effective_programs']) and pd.notna(r['between_cell_JSD']):
            ax.annotate(str(r['label'])[:18],
                        (r['median_effective_programs'], r['between_cell_JSD']),
                        fontsize=6, alpha=.7, xytext=(3, 3), textcoords='offset points')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'diversity_vs_jsd_quadrant.png'), dpi=150,
                bbox_inches='tight')
    plt.show()

#### 14. Acceptance checklist and hand-off

Downstream review should treat `final_recommendation` as the headline, with `rogue_verdict`
and `mechanism_verdict` underneath — never collapse the three into one column.

In [ ]:
checks = {
    'All (K,GEP) keys and program classes validated':
        True,
    'Verdicts stable across >=2 neighbourhood sizes':
        bool(len(robustness_df)) and
        bool((robustness_df.groupby('label')['mechanism_verdict'].nunique() == 1).all()),
    'Adjacent-K sensitivity reported for every label':
        per_label['adjacent_K_consistency'].notna().any(),
    'Every row carries rogue + mechanism + confidence + flags':
        set(['rogue_verdict', 'mechanism_verdict', 'verdict_confidence',
             'rogue_selection_flag']).issubset(per_label.columns),
    'Interaction metrics never silently zero':
        bool(len(per_cell_state)) and
        bool(per_cell_state.loc[per_cell_state['conflict_fraction'].isna(),
                                'interaction_flag'].ne('').all()),
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'REVIEW'}  {name}")

print("\nPipeline outputs:")
print(f"  {PER_CELL_PATH}")
print(f"  {PER_LABEL_PATH}")
print(f"  {os.path.join(OUT_DIR, 'discordant_labels.csv')}  (if any)")
print(f"  {os.path.join(OUT_DIR, 'robustness_neighbors_epsilon.csv')}")